In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["XLA_FLAGS"] = (
    "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
)
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREAD"] = "1"
os.environ["NPROC"] = "1"

In [2]:
import computegraph as cg

cg.jaxify.set_using_jax(True)
cg.jaxify.get_using_jax()

True

In [3]:
import jax

In [4]:
#from proto import *
#from engine import *
#from utils import *
#from runners import CompartmentalModel

# from experimental import *
#from managed import *

#from categories import *

import pandas as pd
import numpy as np
import datetime as dt

pd.options.plotting.backend = "plotly"

#import computegraph as cg

from graph import *


In [5]:
from epi import *

In [6]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

array([Compartment :[(Stratification: disease_state, 'S')],
       Compartment :[(Stratification: disease_state, 'I')],
       Compartment :[(Stratification: disease_state, 'R')]], dtype=object)

In [7]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))
loc_strat = humans.stratify(Stratification("location", ["N", "S", "E", "W"]))

In [8]:
from utils import get_strat_prop_dicts

spd = get_strat_prop_dicts(humans)

In [9]:
cvals.indices["compartment"].index.compartments

NameError: name 'cvals' is not defined

In [ ]:
class LazyLookupArray:
    def __init__(self, dims):
        self.dims = dims
        self.lookup = np.zeros([len(d.strata) for d in dims])
        self.values = jnp.array((0.0,))

In [ ]:
ll = LazyLookupArray([age_strat, loc_strat])
ll.lookup, ll.values



(array([[0., 0., 0., 0.],
        [0., 0., 0., 0.]]),
 Array([0.], dtype=float64))

In [10]:
n_spd = get_strat_prop_dicts(cvals.query(loc_strat["N"]).indices["compartment"].index)

NameError: name 'cvals' is not defined

In [11]:
age_cats

NameError: name 'age_cats' is not defined

In [12]:
A = jnp.array([1.5,1.2])
B = jnp.concat([A[np.array((0,))] * 2.0, A])
z = jnp.array((2.0,3.0))

print(B)

B[np.array((1,1))]

[3.  1.5 1.2]


Array([1.5, 1.5], dtype=float64)

In [13]:
shape = (256, 11 , 15)
slen = np.prod(shape)
x = jnp.linspace(0.0, slen, slen, endpoint=False).reshape(shape)

In [14]:
class SliceIndexer:
    @classmethod
    def __getitem__(cls, k):
        return slice_indices(k)

In [15]:
SI = SliceIndexer()

In [16]:
x[0:12:4, 2:9:3, SI[3:5]].sum()

NameError: name 'slice_indices' is not defined

In [17]:
x[SI[0:12:4], 2:9:3, 3:5].sum()

NameError: name 'slice_indices' is not defined

In [18]:
def slice_indices(s):
    return np.arange(s.start, s.stop, s.step)

In [19]:
slice_indices(np.s_[0:16:2])

array([ 0,  2,  4,  6,  8, 10, 12, 14])

In [20]:
@jit
def thing(x):
    return x[0:16:2, np.array(([0,5,4,1,1])), 4].sum()

In [21]:
@jit
def thing2(x):
    xsub0 = x[SI[0:16:2]]
    xsub1 = xsub0[:, np.array(([0,5,4,1,1]))]
    xsub2 = xsub1[:, :, 4]
    return xsub2.sum()

In [22]:
y = np.arange(slen).reshape(shape)
flatidx = y[0:16:2, np.array(([0,5,4,1,1])), 4].flatten()

In [29]:
def thing3(x):
    return jnp.sum(x.flatten()[flatidx])

In [24]:
def thingstate(inx, i):
    upd = inx/thing3(inx)
    return upd, upd

res, _  = lax.scan(thingstate, x, length=1000)


In [55]:
def thingscan(x, steps):
    def thingstate(inx, i):
        upd = jnp.add(inx, thing3(inx)*0.000001)
        return upd, None

    res, last  = lax.scan(thingstate, x, length=steps)
    return res[0,0,0]

thingscan = jit(thingscan, static_argnums=[1])


In [57]:
%time yres = [np.array(thingscan(x, 200000))]

CPU times: total: 13.3 s
Wall time: 5.15 s


In [102]:
yres[0]

array(63883.788707)

In [84]:
thingscan(x*4, 1000000)

KeyboardInterrupt: 

In [33]:
%%time
res = thingscan(x*4, 1000000)
y = np.sum(res)

CPU times: total: 0 ns
Wall time: 329 μs


In [69]:
y

Array(13.28607383, dtype=float64)

In [234]:
x2 = x/thing2(x)
(x2 / thing2(x2)).sum(), x2.sum()

(Array(18709.89261745, dtype=float64), Array(18709.89261745, dtype=float64))

In [219]:
make_jaxpr(thing2)(x)

{ lambda ; a:f64[256,11,15]. let
    b:f64[8,5] = pjit[
      name=thing2
      jaxpr={ lambda c:i64[8] d:i64[5]; e:f64[256,11,15]. let
          f:i64[8] = device_put[
            copy_semantics=[<CopySemantics.ALIAS: 1>]
            devices=[None]
            srcs=[None]
          ] c
          g:bool[8] = lt f 0
          h:i64[8] = add f 256
          i:i64[8] = select_n g f h
          j:i32[8] = convert_element_type[new_dtype=int32 weak_type=False] i
          k:i32[8,1] = broadcast_in_dim[
            broadcast_dimensions=(0,)
            shape=(8, 1)
            sharding=None
          ] j
          l:f64[8,11,15] = gather[
            dimension_numbers=GatherDimensionNumbers(offset_dims=(1, 2), collapsed_slice_dims=(0,), start_index_map=(0,), operand_batching_dims=(), start_indices_batching_dims=())
            fill_value=None
            indices_are_sorted=False
            mode=GatherScatterMode.PROMISE_IN_BOUNDS
            slice_sizes=(1, 11, 15)
            unique_indices

In [164]:
thing2(x).shape

(256, 3)

In [143]:
make_jaxpr(thing2)(x)

ValueError: Incompatible shapes for broadcasting: shapes=[(8,), (5,), ()]

In [142]:
make_jaxpr(thing)(x)

{ lambda ; a:f64[256,11,3]. let
    b:f64[8,5] = pjit[
      name=thing
      jaxpr={ lambda c:i64[5]; d:f64[256,11,3]. let
          e:i64[5] = device_put[
            copy_semantics=[<CopySemantics.ALIAS: 1>]
            devices=[None]
            srcs=[None]
          ] c
          f:bool[5] = lt e 0
          g:i64[5] = add e 11
          h:i64[5] = select_n f e g
          i:bool[] = lt 4 0
          j:i64[] = add 4 3
          k:i64[] = select_n i 4 j
          l:i32[8] = iota[dimension=0 dtype=int32 shape=(8,) sharding=None] 
          m:i32[8] = mul 2 l
          n:i32[8] = add 0 m
          o:i64[5] = broadcast_in_dim[
            broadcast_dimensions=()
            shape=(5,)
            sharding=None
          ] k
          p:i32[5] = convert_element_type[new_dtype=int32 weak_type=False] h
          q:i32[5] = convert_element_type[new_dtype=int32 weak_type=False] o
          r:i32[8,5,1] = broadcast_in_dim[
            broadcast_dimensions=(0,)
            shape=(8, 5, 1)
  

In [ ]:
# [child, adult] = x[1.0,1.2]
# :
# age
# A: [child, adult]
#      [x[0] , x[1] ]
# adj_with (child,N) * y=2.0
# ->
# simplest approach may just be to fully expand combinations
# age: [child, child, child, child, adult, adult, adult, adult] 
# loc: [N,     S,     E,     W,   , N    , S,   , E    , W]
#      [x[0]*y,x[0],  x[0],  x[0] , x[1] , x[1],  x[1]   x[1]]
# 
# Strict encompassing; A strictly encompasses B iff there are no
# compartments with traits from B that do not also have a trait of A
# (or): iff all compartments in B have an A trait
#
# Non-strict encompassing example
# 
#     S [I_asymp] [I_mod] [I_severe] R
#  C
#  A

#      asymp mod severe
#  C   
#  A

#     S [I_asymp] ... R 
#  C
#  A

# loc * age * inf_status
# 

#     C A
#  N  2 1 
#  S  0 1
#  E  0 1
#  W  0 1
# adj_with [loc: S,W] z[2.0, 3.0]

#     C A
#  N  2 1 
#  S  3 5
#  E  0 1
#  W  4 6

# A.at[]

# 2darray [age,loc]
# [age*workrisk, loc]
#
#
# vs compact representation
# B:
# [child, N], [child[E, S, W]], [adult]
# x[0]*y    , x[0]            , x[1]
#
# adj_with [loc: S,W] z[2.0, 3.0]
#
# C_full: 
# 
# C:
# [child, N], [child[E]], [child[S]], [child[W]], [adult, [N, E]], [adult, S], adult[W]
# _           _           z[0]* _     z[1]*_      _                z[0] * _  , z[1] * _

age_spd = {age_strat: np.array(("child","adult"))}
x = np.array((1.2, 1.5))

lookups = {"x": x}

class LookupRef:
    def __init__(self, src, idx):
        self.src = src
        self.idx = idx

    def __repr__(self):
        return f"LU({self.src}[{self.idx}])"




list(itertools.product(age_spd[age_strat], [0,1]))

[(np.str_('child'), 0),
 (np.str_('child'), 1),
 (np.str_('adult'), 0),
 (np.str_('adult'), 1)]

In [103]:
set([LookupRef("x", 0),LookupRef("x", 1)])

{LU(x[0]), LU(x[1])}

In [101]:
lexp = LazyExpandingArray(n_spd, humans)

In [85]:
humans.stratifications

{Stratification: disease_state: None,
 Stratification: age: (Stratification: disease_state, ('S', 'I', 'R')),
 Stratification: location: (Stratification: disease_state, ('S', 'I', 'R'))}

In [ ]:
# Strictly encompassing stratifications
# Do all product combinations of 2 stratifications exist in this model?
# Counterexamples
# work_risk_group stratifies age["adult"]
# so work_risk_group does not strictly encompass age_strat
# ie (age["child"], work_risk_group["high"]) is invalid
# For now we just work with the assumption that everything applies to everything
# maps to everything - this will break soon (eg severity does not strictly encompass disease_state)

In [ ]:
for cat in age_cats:
    for trait in cat.traits: # traitgroups? these are treated as "traita" or "traitb" or...
        print(cat, trait)
        for strat, trait_values in trait:
            lexp.spd[strat.name]


Category: [(Stratification: age, ['child'])] (Stratification: age, ['child'])
Category: [(Stratification: age, ['adult'])] (Stratification: age, ['adult'])


In [22]:
mm_data = np.array([
        [2.0, 1.0],
        [0.0, 0.0]
    ])

In [61]:
age_cats = age_strat.categories()
infectees = age_cats
infectors = age_cats

mm = mixing_matrix(mm_data, infectors, infectees)
mm.indices["dest"].index

CategoryGroup
Category: [(Stratification: age, ['child'])]
Category: [(Stratification: age, ['adult'])]

In [24]:
class Thing:
    def __matmul__(self, other):
        return "shoes"

    def __rmatmul__(self, other):
        return "bats"

In [25]:
t = Thing()

5 @ t

'bats'

In [26]:
iprocess = defer(InfectionProcess)(mm, infectees, infectors, disease_state["I"])

In [27]:
foi = defer(InfectionProcess.process)(
    iprocess, CompartmentValues, Parameter("contact_rate", 0.2)
)

In [28]:
infection = TransitionFlow("infection", disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow("recovery",
    disease_state["I"], disease_state["R"], Parameter("recovery_rate", 0.1)
)

In [29]:
infection.adjustments.append(CategoryData(loc_strat.categories(), np.array([0.0, 1.0, 0.5, 1.1])))

In [30]:
times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)

epi_model.add_flow(infection)
epi_model.add_flow(recovery)

In [31]:
pop_data = pd.Series(index=["N","E","S","W"], 
                     data=np.array([1000.0,1500.0,200.0,500.0]))

base_pops = strat_data_from_pandas(pop_data, loc_strat)
#base_pops = cat_data_from_pandas(df, loc_strat.categories().product(age_strat.categories()),"pop")
pop_splits = [
    CategoryData(disease_state.categories(), jnp.array(([0.9,0.1,0.0])))
]

epi_model.set_initial_population(base_pops, pop_splits)


In [34]:
cvals = build_istate(humans, base_pops, pop_splits)

In [45]:
def ma_matmul():
    pass

cv_age = cvals.query(disease_state["I"]).sumcats(age_strat.categories())

In [56]:
cv_ns = cvals.sumcats(loc_strat.categories()[0:2])

In [59]:
cv_ns.reconcile(cv_age)[1].indices

{'category': ManagedCategoryGroupIndex: maps [category]
 CategoryGroup
 Category: [(Stratification: age, ['child'])]
 Category: [(Stratification: age, ['adult'])]}

In [54]:
(cvals.sumcats(loc_strat.categories()[0:2]) * cv_age).indices

{'category': ManagedCategoryGroupIndex: maps [category]
 CategoryGroup
 Category: [(Stratification: location, ['N'])]
 Category: [(Stratification: location, ['S'])]}

In [47]:
cv_age

ManagedArray
['category'] (2,)
Indices:
['category']
Data:
[160. 160.]

In [46]:
cv_age * cv_age

ManagedArray
['category'] (2,)
Indices:
['category']
Data:
[25600. 25600.]

In [ ]:
class ManagedMatrix:
    def __init__(self, ma):
        self.ma = ma

    def __matmul__

ManagedArray
['dest', 'source'] (2, 2)
Indices:
['dest', 'source']
Data:
[[2. 1.]
 [0. 0.]]

In [42]:
cv_age * cvals

TypeError: ('Unsupported type', {'category': ManagedCategoryGroupIndex: maps [category]
CategoryGroup
Category: [(Stratification: age, ['child'])]
Category: [(Stratification: age, ['adult'])]})

In [ ]:
params = {"contact_rate": 0.2, "recovery_rate": 0.01}

results = epi_model.run(params)

In [ ]:
compres = results["compartments"]
flowres = results["flows"]

In [ ]:
compres

In [ ]:
compres.sumcats(compartment=loc_strat.categories()).to_pandas_df().plot()

In [ ]:
#type: ignore
non_infectious = Category(disease_state["S","R"]) 
infectious = Category(disease_state["I"])
inf_status_groups = CategoryGroup([non_infectious,infectious]).product(age_strat.categories())

compres.sumcats(compartment=inf_status_groups).to_pandas_df().plot()

In [ ]:
infres = flowres["infection"]
infres

In [ ]:
infres.to_pandas_df().plot()

In [ ]:
# Zoom in and show only age cats
infres.query(time=np.s_[:"aug 1 1980"]).sumcats(dest=age_cats).to_pandas_df().plot()

In [ ]:
mm

In [ ]:
mm.query(dest=age_strat["adult"])

In [ ]:
results["compartments"].sumcats(compartment=disease_state.categories()).to_pandas_df().plot()